In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from matplotlib import pyplot as plt
import joblib

Load data

In [ ]:
# 1. Carregar os dados
train_data = pd.read_csv('dataset_area/final.csv')
test_data = pd.read_csv('dataset_area/final_test.csv')

target = 'Delta_area'
group_col = 'AnomalyID'

# !!! PASSO CRUCIAL PARA O TIMESERIES_SPLIT !!!
# Temos de ordenar tudo por DATA primeiro. Assim, cada corte do Split vai apanhar
# o mesmo semestre (ex: 2016-07) para TODAS as anomalias ao mesmo tempo no treino.
train_data['data'] = pd.to_datetime(train_data['data'])
test_data['data'] = pd.to_datetime(test_data['data'])

train_data = train_data.sort_values(by='data').reset_index(drop=True)

cat_features = ['AnomalyID', 'FacadeRoofID', 'AnomalyType', 'Figura_name']
date_cols = ['data']

# Processamento de datas (mantendo apenas o ano e mês para não criar colunas vazias)
for df in [train_data, test_data]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])
        df[f'{col}_year'] = df[col].dt.year
        df[f'{col}_month'] = df[col].dt.month
    df.drop(columns=date_cols, inplace=True)

# Separar X e y
X_train_full = train_data.drop(columns=[target])
y_train_full = train_data[target]

X_test = test_data.drop(columns=[target])
y_test = test_data[target]

PARAMS = {
    'iterations': 5000,
    'learning_rate': 0.01,
    'depth': 8,
    'loss_function': 'RMSE', # RMSE ajuda o modelo a não se agarrar apenas ao zero
    'eval_metric': 'MAE',
    'random_seed': 42,
    'verbose': False
}

# Com 45 linhas totais após o drop_duplicates, 3 ou 4 splits é o ideal para ter dados suficientes em cada bloco
gkf = TimeSeriesSplit(n_splits=4)

cv_mae = []
cv_r2 = []

print("=== CORRENDO VALIDAÇÃO TEMPORAL (TIMESERIES SPLIT) ===")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_full, y_train_full)):
    X_fold_train = X_train_full.iloc[train_idx]
    X_val = X_train_full.iloc[val_idx]
    y_fold_train = y_train_full.iloc[train_idx]
    y_val = y_train_full.iloc[val_idx]

    model = CatBoostRegressor(**PARAMS)

    model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=150,
        verbose=False
    )

    preds = model.predict(X_val)
    cv_mae.append(mean_absolute_error(y_val, preds))
    cv_r2.append(r2_score(y_val, preds))

    print(f'Fold {fold+1} -> MAE: {cv_mae[-1]:.4f} | R²: {cv_r2[-1]:.4f}')

print('\n=== CRITÉRIO GLOBAL CROSS VALIDATION ===')
print(f'MAE médio: {np.mean(cv_mae):.4f}')
print(f'R² médio: {np.mean(cv_r2):.4f}')

print('\n=== TREINANDO MODELO FINAL CONTRA O TESTE ===')
final_model = CatBoostRegressor(**PARAMS)

final_model.fit(
    X_train_full,
    y_train_full,
    cat_features=cat_features,
    verbose=100
)

test_preds = final_model.predict(X_test)
test_preds = np.clip(test_preds, 0, None)
test_mae = mean_absolute_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)

print('\n=== TESTE FINAL (RESULTADOS) ===')
print(f'MAE: {test_mae:.4f}')
print(f'R²: {test_r2:.4f}')

In [ ]:
comparacao = pd.DataFrame({
    'AnomalyID': test_data['AnomalyID'].values,
    'Area': test_data['prev_area'].values,
    'Delta_Real': y_test.values,
    'Delta_Previsto': test_preds,
    'Erro_Abs_Delta': abs(y_test.values - test_preds),

    'Area_Final_Real': test_data['prev_area'].values + y_test.values,
    'Area_Final_Prevista': test_data['prev_area'].values + test_preds
})

print(comparacao.head(1000))
comparacao.to_csv('Result_final.csv', index=False)

# Calcular as somas globais das áreas absolutas
area_total_final_prevista = comparacao['Area_Final_Prevista'].sum()
area_total_final_real = comparacao['Area_Final_Real'].sum()

print("\n=== ÁREA FINAL ABSOLUTA ACUMULADA (ÁREA + DELTA) ===")
print(f"Área Total Final Prevista: {area_total_final_prevista:.4f} cm²")
print(f"Área Total Final Real:     {area_total_final_real:.4f} cm²")

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, test_preds)
plt.xlabel("Real")
plt.ylabel("Previsto")
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()]
)
plt.show()

In [ ]:
# Save model
joblib.dump(final_model, "model/area_model.pkl")